# TriFlowQC Tutorial: Qutrit Neural Network Training

This notebook demonstrates building and training Quantum Neural Networks with qutrits.

## Contents
1. Data Preparation
2. Building a QNN Model
3. Training the Model
4. Evaluation and Analysis
5. Advanced: Optimizer Comparison
6. Advanced: Noise-Aware Training

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline

from triflowqc.nn.layers.embedding import AngleEmbedding
from triflowqc.nn.layers.dense import DenseQutrit
from triflowqc.nn.layers.entangling import EntanglingLayer
from triflowqc.nn.layers.measurement import MeasurementLayer
from triflowqc.nn.models.sequential import Sequential
from triflowqc.nn.losses import MSELoss, CrossEntropyLoss
from triflowqc.nn.metrics import Accuracy
from triflowqc.nn.callbacks import EarlyStopping, ModelCheckpoint
from triflowqc.optim.adam import Adam
from triflowqc.optim.sgd import SGD
from triflowqc.optim.spsa import SPSA
from triflowqc.data.loader import DataLoader
from triflowqc.data.encoder import AngleEncoder, AmplitudeEncoder
from triflowqc.data.preprocessor import Preprocessor

np.random.seed(42)
print('TriFlowQC Neural Network modules loaded!')

## 1. Data Preparation

We'll create a synthetic 3-class classification problem that naturally maps to qutrits.

In [ ]:
# Generate synthetic 3-class data
n_samples_per_class = 20
n_features = 3

# Class 0: centered at (1, 0, 0)
X0 = np.random.randn(n_samples_per_class, n_features) * 0.3 + [1, 0, 0]
# Class 1: centered at (0, 1, 0)
X1 = np.random.randn(n_samples_per_class, n_features) * 0.3 + [0, 1, 0]
# Class 2: centered at (0, 0, 1)
X2 = np.random.randn(n_samples_per_class, n_features) * 0.3 + [0, 0, 1]

X = np.vstack([X0, X1, X2])
y_labels = np.array([0]*n_samples_per_class + [1]*n_samples_per_class + [2]*n_samples_per_class)
Y = np.eye(3)[y_labels]  # One-hot encoding

print(f'Dataset: {X.shape[0]} samples, {X.shape[1]} features, 3 classes')
print(f'X shape: {X.shape}')
print(f'Y shape: {Y.shape}')

# Visualize
fig = plt.figure(figsize=(8, 6))
ax = fig.add_subplot(111, projection='3d')
colors = ['#e74c3c', '#3498db', '#2ecc71']
for cls in range(3):
    mask = y_labels == cls
    ax.scatter(X[mask, 0], X[mask, 1], X[mask, 2], 
               c=colors[cls], label=f'Class {cls}', alpha=0.7)
ax.set_xlabel('Feature 0')
ax.set_ylabel('Feature 1')
ax.set_zlabel('Feature 2')
ax.legend()
plt.title('Synthetic 3-Class Dataset')
plt.show()

In [ ]:
# Preprocess
preprocessor = Preprocessor(method='standard')
X_scaled = preprocessor.fit_transform(X)
print(f'Scaled mean: {X_scaled.mean(axis=0).round(4)}')
print(f'Scaled std: {X_scaled.std(axis=0).round(4)}')

# Encode for quantum circuit
encoder = AngleEncoder()
X_encoded = np.array([encoder.encode(x) for x in X_scaled])
print(f'\nEncoded range: [{X_encoded.min():.3f}, {X_encoded.max():.3f}]')

# Train/test split
n_total = len(X_encoded)
indices = np.random.permutation(n_total)
split = int(0.8 * n_total)
train_idx, test_idx = indices[:split], indices[split:]

X_train, X_test = X_encoded[train_idx], X_encoded[test_idx]
Y_train, Y_test = Y[train_idx], Y[test_idx]
y_test_labels = y_labels[test_idx]

print(f'\nTraining: {len(X_train)} samples')
print(f'Testing: {len(X_test)} samples')

## 2. Building a QNN Model

The model architecture follows a Keras-like Sequential API:

1. **AngleEmbedding**: Encodes classical data as rotation angles
2. **DenseQutrit**: Parameterized single-qutrit rotations
3. **EntanglingLayer**: CSUM gates for inter-qutrit entanglement
4. **MeasurementLayer**: Extracts class probabilities from measurement

In [ ]:
n_qutrits = n_features  # 1 qutrit per feature
n_classes = 3

model = Sequential([
    AngleEmbedding(n_qutrits=n_qutrits),
    DenseQutrit(n_qutrits=n_qutrits, n_layers=2, rotation_type='basic'),
    EntanglingLayer(n_qutrits=n_qutrits, pattern='linear'),
    DenseQutrit(n_qutrits=n_qutrits, n_layers=1, rotation_type='basic'),
    EntanglingLayer(n_qutrits=n_qutrits, pattern='ring'),
    MeasurementLayer(n_qutrits=n_qutrits, n_classes=n_classes),
])

model.summary()
print(f'\nTotal trainable parameters: {len(model.parameters())}')

In [ ]:
# Test forward pass
x_sample = X_train[0]
output = model.predict(x_sample)
print(f'Input: {x_sample.round(3)}')
print(f'Output probabilities: {np.array(output).round(4)}')
print(f'Predicted class: {np.argmax(output)}')

## 3. Training the Model

The training loop uses:
- **Parameter-shift rule** for gradient computation (shifted by ±2π/3 for qutrits)
- **Adam optimizer** for parameter updates
- **MSE loss** for multi-class classification

In [ ]:
# Compile model
optimizer = Adam(learning_rate=0.05)
loss_fn = MSELoss()
model.compile(optimizer=optimizer, loss=loss_fn)

# Train
print('Training...')
history = model.fit(
    X_train, Y_train,
    epochs=15,
    verbose=True,
)

print(f'\nFinal loss: {history["loss"][-1]:.6f}')

In [ ]:
# Plot training history
plt.figure(figsize=(10, 4))
plt.plot(range(1, len(history['loss'])+1), history['loss'], 'o-', linewidth=2)
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('Training Loss')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 4. Evaluation

Test the trained model on held-out data.

In [ ]:
# Predict on test set
predictions = []
pred_probs = []
for x in X_test:
    output = model.predict(x)
    pred_probs.append(output)
    predictions.append(np.argmax(output))

predictions = np.array(predictions)
pred_probs = np.array(pred_probs)

accuracy = np.mean(predictions == y_test_labels)
print(f'Test Accuracy: {accuracy:.2%}')

# Per-class accuracy
for cls in range(3):
    mask = y_test_labels == cls
    if mask.sum() > 0:
        cls_acc = np.mean(predictions[mask] == cls)
        print(f'  Class {cls}: {cls_acc:.2%} ({mask.sum()} samples)')

In [ ]:
# Confusion matrix
confusion = np.zeros((3, 3), dtype=int)
for true, pred in zip(y_test_labels, predictions):
    confusion[true, pred] += 1

plt.figure(figsize=(6, 5))
plt.imshow(confusion, cmap='Blues', interpolation='nearest')
plt.colorbar(label='Count')
for i in range(3):
    for j in range(3):
        plt.text(j, i, str(confusion[i, j]), ha='center', va='center', fontsize=16,
                color='white' if confusion[i, j] > confusion.max()/2 else 'black')
plt.xlabel('Predicted')
plt.ylabel('True')
plt.title('Confusion Matrix')
plt.xticks([0, 1, 2])
plt.yticks([0, 1, 2])
plt.tight_layout()
plt.show()

## 5. Optimizer Comparison

Compare SGD, Adam, and SPSA optimizers on the same task.

In [ ]:
optimizers = {
    'SGD (lr=0.1)': SGD(learning_rate=0.1),
    'Adam (lr=0.05)': Adam(learning_rate=0.05),
    'SPSA (lr=0.1)': SPSA(learning_rate=0.1),
}

histories = {}
n_epochs = 10

for name, opt in optimizers.items():
    print(f'\nTraining with {name}...')
    np.random.seed(42)  # Same initialization
    m = Sequential([
        AngleEmbedding(n_qutrits=n_qutrits),
        DenseQutrit(n_qutrits=n_qutrits, n_layers=2, rotation_type='basic'),
        EntanglingLayer(n_qutrits=n_qutrits, pattern='linear'),
        MeasurementLayer(n_qutrits=n_qutrits, n_classes=n_classes),
    ])
    m.compile(optimizer=opt, loss=MSELoss())
    h = m.fit(X_train[:10], Y_train[:10], epochs=n_epochs, verbose=False)
    histories[name] = h['loss']
    print(f'  Final loss: {h["loss"][-1]:.6f}')

# Plot comparison
plt.figure(figsize=(10, 5))
for name, losses in histories.items():
    plt.plot(range(1, len(losses)+1), losses, 'o-', label=name, linewidth=2)
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('Optimizer Comparison')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 6. Parameter Analysis

Inspect the learned parameters of the model.

In [ ]:
# Get all parameters
params = model.parameters()
param_values = np.array([p.value for p in params])

print(f'Number of parameters: {len(params)}')
print(f'Parameter range: [{param_values.min():.4f}, {param_values.max():.4f}]')
print(f'Parameter mean: {param_values.mean():.4f}')
print(f'Parameter std: {param_values.std():.4f}')

# Histogram
plt.figure(figsize=(10, 4))
plt.hist(param_values, bins=30, color='steelblue', alpha=0.8, edgecolor='white')
plt.xlabel('Parameter Value')
plt.ylabel('Count')
plt.title('Distribution of Learned Parameters')
plt.grid(True, alpha=0.3, axis='y')
plt.tight_layout()
plt.show()

In [ ]:
# Gradient landscape for first 2 parameters
from triflowqc.optim.parameter_shift import parameter_shift_gradient

def cost_fn(new_params):
    """Evaluate cost with modified parameters."""
    # Save original
    orig = [p.value for p in params]
    # Set new
    for p, v in zip(params, new_params):
        p.value = v
    # Compute loss
    total_loss = 0
    for x, y in zip(X_train[:5], Y_train[:5]):
        pred = model.predict(x)
        total_loss += loss_fn(np.array(pred), y)
    # Restore
    for p, v in zip(params, orig):
        p.value = v
    return total_loss / 5

# Compute gradient
current = np.array([p.value for p in params])
grads = parameter_shift_gradient(cost_fn, current)

plt.figure(figsize=(10, 4))
plt.bar(range(len(grads)), grads, color='steelblue', alpha=0.8)
plt.xlabel('Parameter Index')
plt.ylabel('Gradient')
plt.title('Parameter Gradients at Current Point')
plt.grid(True, alpha=0.3, axis='y')
plt.tight_layout()
plt.show()

print(f'Gradient norm: {np.linalg.norm(grads):.6f}')
print(f'Max |gradient|: {np.max(np.abs(grads)):.6f}')

## Summary

In this tutorial, you learned:
- **Data preparation**: Preprocessing, encoding, and splitting for QNN training
- **Model building**: Sequential API with layers (embedding, dense, entangling, measurement)
- **Training**: Parameter-shift gradients with Adam/SGD/SPSA optimizers
- **Evaluation**: Accuracy, confusion matrix, and parameter analysis
- **Optimizer comparison**: Different optimizers have different convergence behavior

### Key Differences from Qubit QNNs:
- Qutrit rotations use **Gell-Mann matrices** (8 generators) instead of Pauli matrices (3)
- Parameter-shift rule uses **±2π/3 shift** instead of ±π/2
- Each qutrit carries **log₂(3) ≈ 1.585 bits** vs. 1 bit per qubit
- **CSUM gate** (controlled sum mod 3) instead of CNOT
- Running on real hardware requires **qutrit-to-qubit compilation** (2 qubits per qutrit)